# Crop Irrigation Need Prediction
## Kaggle Playground Series - Season 6, Episode 4

**Objective:** Predict the irrigation need (Low, Medium, High) for crops based on environmental and agricultural features.

**Evaluation Metric:** Balanced Accuracy

**Target Score:** 99%+

### Pipeline Overview
1. Data Loading & Initial Inspection
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing & Feature Engineering
4. Model Training & Cross-Validation
5. Model Evaluation & Comparison
6. Ensemble & Final Predictions
7. Submission Generation

## 0. Setup & Imports

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.metrics import (
    balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, accuracy_score
)
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, VotingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone as sk_clone

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

%matplotlib inline

print("All libraries imported successfully!")

In [ ]:
# Configuration
DATA_DIR = os.path.join('..', 'Data')
OUTPUT_DIR = os.path.join('..', 'outputs')
PLOTS_DIR = os.path.join(OUTPUT_DIR, 'plots')
MODELS_DIR = os.path.join(OUTPUT_DIR, 'models')

os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

RANDOM_STATE = 42
N_SPLITS = 5

# Plot Style Setup
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams.update({
    'figure.figsize': (12, 8),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

print("Configuration set!")
print(f"Data directory: {os.path.abspath(DATA_DIR)}")

---
## 1. Data Loading & Initial Inspection

Load the train, test, and sample submission datasets. Perform initial quality checks.

In [ ]:
# Load datasets
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")
print(f"Sample submission shape: {sample_sub.shape}")

In [ ]:
# Data types and column overview
print("--- Column Data Types ---")
print(train.dtypes)
print(f"\nTotal columns: {train.shape[1]}")

In [ ]:
# First few rows
train.head(10)

In [ ]:
# Statistical summary
train.describe()

In [ ]:
# Categorical columns summary
train.describe(include='object')

In [ ]:
# Missing values check
print("--- Missing Values (Train) ---")
missing_train = train.isnull().sum()
print(missing_train[missing_train > 0] if missing_train.sum() > 0 else "No missing values!")

print("\n--- Missing Values (Test) ---")
missing_test = test.isnull().sum()
print(missing_test[missing_test > 0] if missing_test.sum() > 0 else "No missing values!")

In [ ]:
# Target distribution
print("--- Target Distribution ---")
print(train['Irrigation_Need'].value_counts())
print()
print(train['Irrigation_Need'].value_counts(normalize=True).round(4))

In [ ]:
# Duplicate rows check
dupes = train.duplicated().sum()
print(f"Number of duplicate rows: {dupes}")

---
## 2. Exploratory Data Analysis (EDA)

Comprehensive visual analysis of features and their relationship with the target variable.

In [ ]:
# Identify feature types
target = 'Irrigation_Need'
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['id']]
cat_cols = train.select_dtypes(include=['object']).columns.tolist()
cat_cols = [c for c in cat_cols if c != target]

print(f"Numeric features ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical features ({len(cat_cols)}): {cat_cols}")

order = ['Low', 'Medium', 'High']
valid_order = [o for o in order if o in train[target].unique()]
if not valid_order:
    valid_order = train[target].unique().tolist()

### 2.1 Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = sns.color_palette("viridis", len(valid_order))

sns.countplot(data=train, x=target, order=valid_order, ax=axes[0], palette=colors)
axes[0].set_title('Target Distribution (Count)', fontweight='bold')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontweight='bold')

train[target].value_counts().plot.pie(
    autopct='%1.1f%%', ax=axes[1], colors=colors, startangle=90
)
axes[1].set_title('Target Distribution (%)', fontweight='bold')
axes[1].set_ylabel('')

fig.suptitle('Irrigation Need - Target Variable Distribution', fontsize=16, fontweight='bold')
fig.tight_layout()
plt.show()

### 2.2 Numeric Feature Distributions

In [ ]:
if numeric_cols:
    n_cols_plot = min(4, len(numeric_cols))
    n_rows = (len(numeric_cols) + n_cols_plot - 1) // n_cols_plot
    fig, axes = plt.subplots(n_rows, n_cols_plot, figsize=(5 * n_cols_plot, 4 * n_rows))
    if n_rows == 1 and n_cols_plot == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    for i, col in enumerate(numeric_cols):
        sns.histplot(data=train, x=col, hue=target, kde=True,
                     ax=axes[i], palette='viridis', alpha=0.6,
                     hue_order=valid_order)
        axes[i].set_title(f'{col}', fontweight='bold')
        axes[i].legend(title='', fontsize=8)
    
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    
    fig.suptitle('Numeric Feature Distributions by Target', fontsize=16, fontweight='bold', y=1.02)
    fig.tight_layout()
    plt.show()

### 2.3 Boxplots per Target Class

In [ ]:
if numeric_cols:
    n_cols_plot = min(4, len(numeric_cols))
    n_rows = (len(numeric_cols) + n_cols_plot - 1) // n_cols_plot
    fig, axes = plt.subplots(n_rows, n_cols_plot, figsize=(5 * n_cols_plot, 4 * n_rows))
    if n_rows == 1 and n_cols_plot == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    for i, col in enumerate(numeric_cols):
        sns.boxplot(data=train, x=target, y=col, order=valid_order,
                    ax=axes[i], palette='viridis')
        axes[i].set_title(f'{col}', fontweight='bold')
    
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    
    fig.suptitle('Feature Boxplots by Irrigation Need', fontsize=16, fontweight='bold', y=1.02)
    fig.tight_layout()
    plt.show()

### 2.4 Correlation Matrix

In [ ]:
if len(numeric_cols) > 1:
    fig, ax = plt.subplots(figsize=(12, 10))
    corr = train[numeric_cols].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, square=True, linewidths=0.5, ax=ax,
                cbar_kws={"shrink": 0.8})
    ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
    fig.tight_layout()
    plt.show()

### 2.5 Categorical Feature Analysis

In [ ]:
if cat_cols:
    n_cat = len(cat_cols)
    fig, axes = plt.subplots(1, max(n_cat, 1), figsize=(6 * max(n_cat, 1), 5))
    if n_cat == 1:
        axes = [axes]
    
    for i, col in enumerate(cat_cols):
        ct = pd.crosstab(train[col], train[target], normalize='index')
        ct_cols = [c for c in valid_order if c in ct.columns]
        if ct_cols:
            ct = ct[ct_cols]
        ct.plot(kind='bar', stacked=True, ax=axes[i],
                colormap='viridis', alpha=0.85)
        axes[i].set_title(f'{col} vs {target}', fontweight='bold')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Proportion')
        axes[i].legend(title=target, fontsize=8)
        axes[i].tick_params(axis='x', rotation=45)
    
    fig.suptitle('Categorical Features vs Target', fontsize=16, fontweight='bold')
    fig.tight_layout()
    plt.show()

### 2.6 Pairplot (Sampled)

In [ ]:
if len(numeric_cols) >= 2:
    sample_size = min(2000, len(train))
    sample_df = train[numeric_cols + [target]].sample(
        sample_size, random_state=RANDOM_STATE
    )
    pair_cols = numeric_cols[:5]  # Limit to 5 features for readability
    g = sns.pairplot(sample_df[pair_cols + [target]], hue=target,
                     hue_order=valid_order, palette='viridis',
                     diag_kind='kde', plot_kws={'alpha': 0.4, 's': 15})
    g.figure.suptitle('Pairplot of Top Features', fontsize=16, fontweight='bold', y=1.02)
    plt.show()
    
print("EDA Complete!")

---
## 3. Data Preprocessing & Feature Engineering

Key preprocessing steps:
- Handle missing values
- Remove duplicate rows
- Encode categorical features
- Engineer new features (interactions, thresholds, polynomials)

In [ ]:
# Save IDs and separate target
train_id = train['id'].copy()
test_id = test['id'].copy()

y = train[target].copy()

# Encode target
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)
print(f"Target classes: {list(le_target.classes_)}")
print(f"Encoded as:    {list(range(len(le_target.classes_)))}")

# Drop id and target
train_features = train.drop(columns=['id', target])
test_features = test.drop(columns=['id'])

# Combine for consistent preprocessing
combined = pd.concat([train_features, test_features], axis=0, ignore_index=True)
n_train = len(train_features)

print(f"\nCombined shape: {combined.shape}")
print(f"Train samples: {n_train}")
print(f"Test samples:  {len(test_features)}")

### 3.1 Handle Missing Values

In [ ]:
for col in combined.columns:
    null_count = combined[col].isnull().sum()
    if null_count > 0:
        if combined[col].dtype in ['float64', 'int64']:
            combined[col].fillna(combined[col].median(), inplace=True)
            print(f"  Filled {col} with median ({null_count} nulls)")
        else:
            combined[col].fillna(combined[col].mode()[0], inplace=True)
            print(f"  Filled {col} with mode ({null_count} nulls)")

if combined.isnull().sum().sum() == 0:
    print("No missing values found!")

### 3.2 Handle Duplicates

In [ ]:
train_part = combined.iloc[:n_train].copy()
train_part['__target__'] = y_encoded
before = len(train_part)
train_part = train_part.drop_duplicates()
after = len(train_part)

if before > after:
    y_encoded = train_part['__target__'].values
    train_part = train_part.drop(columns=['__target__'])
    combined = pd.concat([train_part, combined.iloc[n_train:]], axis=0, ignore_index=True)
    n_train = len(train_part)
    print(f"Removed {before - after} duplicate rows from training data")
else:
    train_part = train_part.drop(columns=['__target__'])
    print(f"No duplicates to remove")

### 3.3 Encode Categorical Features

In [ ]:
actual_cat_cols = combined.select_dtypes(include=['object']).columns.tolist()
actual_num_cols = combined.select_dtypes(include=[np.number]).columns.tolist()

label_encoders = {}
for col in actual_cat_cols:
    le = LabelEncoder()
    combined[col] = le.fit_transform(combined[col].astype(str))
    label_encoders[col] = le
    print(f"Encoded {col}: {list(le.classes_)} -> {list(range(len(le.classes_)))}")

### 3.4 Feature Engineering

Create interaction features, threshold features, polynomial features, and categorical combinations.
Key insight: The target is driven by rule-based logic involving soil moisture, temperature, rainfall thresholds.

In [ ]:
original_features = combined.columns.tolist()

# Detect key columns dynamically
soil_col = [c for c in combined.columns if 'soil' in c.lower() and 'moisture' in c.lower()]
temp_col = [c for c in combined.columns if 'temp' in c.lower()]
rain_col = [c for c in combined.columns if 'rain' in c.lower()]
wind_col = [c for c in combined.columns if 'wind' in c.lower()]
growth_col = [c for c in combined.columns if 'growth' in c.lower() or 'stage' in c.lower()]
mulch_col = [c for c in combined.columns if 'mulch' in c.lower()]
humidity_col = [c for c in combined.columns if 'humid' in c.lower()]
crop_col = [c for c in combined.columns if 'crop' in c.lower() and 'type' in c.lower()]

fe_count = 0

# Interaction features
if soil_col and temp_col:
    sc, tc = soil_col[0], temp_col[0]
    combined['soil_temp_interaction'] = combined[sc] * combined[tc]
    combined['soil_temp_ratio'] = combined[sc] / (combined[tc] + 1e-6)
    fe_count += 2

if soil_col and rain_col:
    sc, rc = soil_col[0], rain_col[0]
    combined['soil_rain_interaction'] = combined[sc] * combined[rc]
    combined['water_availability'] = combined[sc] + combined[rc]
    fe_count += 2

if temp_col and rain_col:
    tc, rc = temp_col[0], rain_col[0]
    combined['evaporation_proxy'] = combined[tc] / (combined[rc] + 1e-6)
    fe_count += 1

if temp_col and wind_col:
    tc, wc = temp_col[0], wind_col[0]
    combined['wind_chill_proxy'] = combined[tc] * combined[wc]
    fe_count += 1

# Soil moisture features (key driver!)
if soil_col:
    sc = soil_col[0]
    combined['soil_moisture_sq'] = combined[sc] ** 2
    combined['soil_moisture_log'] = np.log1p(combined[sc])
    combined['soil_low'] = (combined[sc] < 25).astype(int)
    combined['soil_medium'] = ((combined[sc] >= 25) & (combined[sc] < 50)).astype(int)
    combined['soil_high'] = (combined[sc] >= 50).astype(int)
    fe_count += 5

# Temperature features
if temp_col:
    tc = temp_col[0]
    combined['temp_sq'] = combined[tc] ** 2
    combined['temp_high'] = (combined[tc] > 30).astype(int)
    combined['temp_low'] = (combined[tc] < 15).astype(int)
    fe_count += 3

# Rainfall features
if rain_col:
    rc = rain_col[0]
    combined['rain_sq'] = combined[rc] ** 2
    combined['rain_log'] = np.log1p(combined[rc])
    combined['no_rain'] = (combined[rc] < 1).astype(int)
    fe_count += 3

# Humidity features
if humidity_col:
    hc = humidity_col[0]
    combined['humidity_sq'] = combined[hc] ** 2
    combined['humidity_high'] = (combined[hc] > 70).astype(int)
    fe_count += 2

# Wind features
if wind_col:
    wc = wind_col[0]
    combined['wind_sq'] = combined[wc] ** 2
    combined['wind_high'] = (combined[wc] > 20).astype(int)
    fe_count += 2

# Categorical combinations
if growth_col and mulch_col:
    gc, mc = growth_col[0], mulch_col[0]
    combined['growth_mulch'] = combined[gc].astype(str) + '_' + combined[mc].astype(str)
    le_gm = LabelEncoder()
    combined['growth_mulch'] = le_gm.fit_transform(combined['growth_mulch'])
    fe_count += 1

if crop_col and growth_col:
    cc, gc = crop_col[0], growth_col[0]
    combined['crop_growth'] = combined[cc].astype(str) + '_' + combined[gc].astype(str)
    le_cg = LabelEncoder()
    combined['crop_growth'] = le_cg.fit_transform(combined['crop_growth'])
    fe_count += 1

# Statistical aggregation features
num_feats = [c for c in actual_num_cols if c in combined.columns]
if len(num_feats) >= 2:
    combined['feat_mean'] = combined[num_feats].mean(axis=1)
    combined['feat_std'] = combined[num_feats].std(axis=1)
    combined['feat_max'] = combined[num_feats].max(axis=1)
    combined['feat_min'] = combined[num_feats].min(axis=1)
    combined['feat_range'] = combined['feat_max'] - combined['feat_min']
    fe_count += 5

new_features = [c for c in combined.columns if c not in original_features]
print(f"Created {fe_count} engineered features:")
for f in new_features:
    print(f"  + {f}")

print(f"\nFinal feature count: {combined.shape[1]}")

### 3.5 Split Back to Train/Test

In [ ]:
X_train = combined.iloc[:n_train].copy().reset_index(drop=True)
X_test = combined.iloc[n_train:].copy().reset_index(drop=True)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_encoded.shape}")

---
## 4. Model Training & Cross-Validation

Training 5 models with Stratified K-Fold cross-validation:
- **LightGBM** - Fast gradient boosting
- **XGBoost** - Extreme gradient boosting
- **CatBoost** - Categorical-aware boosting
- **ExtraTrees** - Extremely randomized trees
- **RandomForest** - Classic ensemble

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# Define models
models = {
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=8,
        num_leaves=63,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=RANDOM_STATE,
        verbose=-1,
        n_jobs=-1,
        class_weight='balanced'
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=8,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        eval_metric='mlogloss',
        use_label_encoder=False,
        n_jobs=-1,
        tree_method='hist'
    ),
    'CatBoost': CatBoostClassifier(
        iterations=2000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=3,
        random_seed=RANDOM_STATE,
        verbose=0,
        auto_class_weights='Balanced',
        eval_metric='TotalF1'
    ),
    'ExtraTrees': ExtraTreesClassifier(
        n_estimators=1000,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight='balanced'
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=1000,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight='balanced'
    ),
}

print(f"Defined {len(models)} models for training")
for name in models:
    print(f"  - {name}")

In [ ]:
# Train all models with cross-validation
results = {}
trained_models = {}
oof_predictions = {}

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"  Training {name}")
    print(f"{'='*60}")
    
    oof_preds = np.zeros(len(X_train))
    oof_proba = np.zeros((len(X_train), len(le_target.classes_)))
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_encoded)):
        X_tr = X_train.iloc[train_idx]
        y_tr = y_encoded[train_idx]
        X_val = X_train.iloc[val_idx]
        y_val = y_encoded[val_idx]
        
        if name == 'LightGBM':
            model_clone = lgb.LGBMClassifier(**model.get_params())
            model_clone.fit(
                X_tr, y_tr,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(50, verbose=False)]
            )
        elif name == 'XGBoost':
            model_clone = xgb.XGBClassifier(**model.get_params())
            model_clone.fit(
                X_tr, y_tr,
                eval_set=[(X_val, y_val)],
                verbose=False
            )
        elif name == 'CatBoost':
            model_clone = CatBoostClassifier(**model.get_all_params())
            model_clone.fit(
                X_tr, y_tr,
                eval_set=(X_val, y_val),
                early_stopping_rounds=50,
                verbose=0
            )
        else:
            model_clone = sk_clone(model)
            model_clone.fit(X_tr, y_tr)
        
        val_preds = model_clone.predict(X_val)
        val_proba = model_clone.predict_proba(X_val)
        
        oof_preds[val_idx] = val_preds
        oof_proba[val_idx] = val_proba
        
        fold_score = balanced_accuracy_score(y_val, val_preds)
        fold_scores.append(fold_score)
        
        print(f"  Fold {fold + 1}: Balanced Accuracy = {fold_score:.6f}")
    
    overall_score = balanced_accuracy_score(y_encoded, oof_preds)
    mean_score = np.mean(fold_scores)
    std_score = np.std(fold_scores)
    
    print(f"  > Mean CV Score:    {mean_score:.6f} +/- {std_score:.6f}")
    print(f"  > Overall OOF Score: {overall_score:.6f}")
    
    results[name] = {
        'mean_cv': mean_score,
        'std_cv': std_score,
        'oof_score': overall_score,
        'fold_scores': fold_scores
    }
    oof_predictions[name] = oof_proba
    
    # Train final model on full training data
    if name == 'LightGBM':
        final_model = lgb.LGBMClassifier(**model.get_params())
    elif name == 'XGBoost':
        final_model = xgb.XGBClassifier(**model.get_params())
    elif name == 'CatBoost':
        final_model = CatBoostClassifier(**model.get_all_params())
    else:
        final_model = sk_clone(model)
    
    final_model.fit(X_train, y_encoded)
    trained_models[name] = final_model

print("\nAll models trained!")

---
## 5. Model Evaluation & Comparison

### 5.1 Results Summary

In [ ]:
results_df = pd.DataFrame({
    name: {
        'Mean CV': f"{res['mean_cv']:.6f}",
        'Std CV': f"{res['std_cv']:.6f}",
        'OOF Score': f"{res['oof_score']:.6f}"
    }
    for name, res in results.items()
}).T

print("Model Comparison:")
results_df

### 5.2 CV Score Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

model_names = list(results.keys())
means = [results[n]['mean_cv'] for n in model_names]
stds = [results[n]['std_cv'] for n in model_names]

colors = sns.color_palette("viridis", len(model_names))
bars = ax.bar(model_names, means, yerr=stds, capsize=5,
              color=colors, alpha=0.85, edgecolor='black', linewidth=0.5)

for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
            f'{mean:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_ylabel('Balanced Accuracy', fontsize=12)
ax.set_title('Model Comparison - Cross-Validation Balanced Accuracy',
             fontsize=16, fontweight='bold')
ax.set_ylim(min(means) - 0.02, 1.0)
ax.axhline(y=0.99, color='red', linestyle='--', alpha=0.7, label='Target: 0.99')
ax.legend()
fig.tight_layout()
plt.show()

### 5.3 Fold Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
fold_data = []
for name, res in results.items():
    for score in res['fold_scores']:
        fold_data.append({'Model': name, 'Balanced Accuracy': score})
fold_df = pd.DataFrame(fold_data)

sns.boxplot(data=fold_df, x='Model', y='Balanced Accuracy', palette='viridis', ax=ax)
sns.stripplot(data=fold_df, x='Model', y='Balanced Accuracy',
              color='black', alpha=0.5, size=6, ax=ax)
ax.set_title('Cross-Validation Fold Score Distribution', fontsize=16, fontweight='bold')
ax.axhline(y=0.99, color='red', linestyle='--', alpha=0.7, label='Target: 0.99')
ax.legend()
fig.tight_layout()
plt.show()

### 5.4 Feature Importance

In [ ]:
best_model_name = max(results, key=lambda x: results[x]['mean_cv'])
best_model = trained_models[best_model_name]
print(f"Best model: {best_model_name} (CV: {results[best_model_name]['mean_cv']:.6f})")

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feature_names = X_train.columns
    imp_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=True).tail(20)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(imp_df['Feature'], imp_df['Importance'],
            color=sns.color_palette("viridis", len(imp_df)))
    ax.set_title(f'Top 20 Feature Importances ({best_model_name})',
                 fontsize=16, fontweight='bold')
    ax.set_xlabel('Importance')
    fig.tight_layout()
    plt.show()

### 5.5 Confusion Matrix

In [ ]:
y_pred_best = best_model.predict(X_train)

fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_encoded, y_pred_best)
disp = ConfusionMatrixDisplay(cm, display_labels=le_target.classes_)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title(f'Confusion Matrix - {best_model_name} (Train)',
             fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

### 5.6 Classification Report

In [ ]:
print(f"Classification Report ({best_model_name}):")
print(classification_report(y_encoded, y_pred_best,
                            target_names=le_target.classes_))

---
## 6. Ensemble & Final Predictions

Create weighted ensemble combining all models, weighted by their CV performance.

In [ ]:
# Weighted Average Ensemble
scores = {name: results[name]['mean_cv'] for name in trained_models}
total = sum(scores.values())
weights = {name: score / total for name, score in scores.items()}

print("Model weights:")
for name, w in weights.items():
    print(f"  {name}: {w:.4f} (CV: {scores[name]:.6f})")

# OOF ensemble evaluation
oof_ensemble_proba = np.zeros((len(X_train), len(le_target.classes_)))
for name, proba in oof_predictions.items():
    oof_ensemble_proba += weights[name] * proba

oof_ensemble_preds = np.argmax(oof_ensemble_proba, axis=1)
ensemble_oof_score = balanced_accuracy_score(y_encoded, oof_ensemble_preds)
print(f"\nWeighted Ensemble OOF Balanced Accuracy: {ensemble_oof_score:.6f}")

In [ ]:
# Generate test predictions
test_proba = np.zeros((len(X_test), len(le_target.classes_)))
for name, model in trained_models.items():
    pred_proba = model.predict_proba(X_test)
    test_proba += weights[name] * pred_proba

test_predictions = np.argmax(test_proba, axis=1)
test_labels = le_target.inverse_transform(test_predictions)

print("Test predictions distribution:")
pred_counts = pd.Series(test_labels).value_counts()
for label, count in pred_counts.items():
    print(f"  {label}: {count} ({count / len(test_labels) * 100:.1f}%)")

In [ ]:
# Top-3 Models Ensemble
sorted_models = sorted(scores.items(), key=lambda x: x[1], reverse=True)
top3 = [name for name, _ in sorted_models[:3]]

top3_total = sum(scores[n] for n in top3)
top3_weights = {n: scores[n] / top3_total for n in top3}

top3_oof_proba = np.zeros((len(X_train), len(le_target.classes_)))
for name in top3:
    top3_oof_proba += top3_weights[name] * oof_predictions[name]

top3_oof_preds = np.argmax(top3_oof_proba, axis=1)
top3_oof_score = balanced_accuracy_score(y_encoded, top3_oof_preds)
print(f"Top-3 Ensemble ({', '.join(top3)}) OOF: {top3_oof_score:.6f}")

# Pick best approach
best_single_name = max(scores, key=scores.get)
best_single_score = scores[best_single_name]

print(f"\n--- Final Score Summary ---")
print(f"Best Single Model ({best_single_name}): {best_single_score:.6f}")
print(f"All-Model Ensemble:                       {ensemble_oof_score:.6f}")
print(f"Top-3 Ensemble:                            {top3_oof_score:.6f}")

# Select best approach
best_score = max(best_single_score, ensemble_oof_score, top3_oof_score)

if best_score == top3_oof_score and top3_oof_score > ensemble_oof_score:
    print(f"\n>> Using Top-3 Ensemble (score: {top3_oof_score:.6f})")
    final_proba = np.zeros((len(X_test), len(le_target.classes_)))
    for name in top3:
        final_proba += top3_weights[name] * trained_models[name].predict_proba(X_test)
    final_preds = np.argmax(final_proba, axis=1)
    final_labels = le_target.inverse_transform(final_preds)
    final_score = top3_oof_score
elif best_score == ensemble_oof_score:
    print(f"\n>> Using All-Model Ensemble (score: {ensemble_oof_score:.6f})")
    final_labels = test_labels
    final_score = ensemble_oof_score
else:
    print(f"\n>> Using Best Single Model: {best_single_name} (score: {best_single_score:.6f})")
    final_preds = trained_models[best_single_name].predict(X_test)
    final_labels = le_target.inverse_transform(final_preds)
    final_score = best_single_score

---
## 7. Submission Generation

In [ ]:
submission = pd.DataFrame({
    'id': test_id,
    'Irrigation_Need': final_labels
})

sub_path = os.path.join(DATA_DIR, 'submission.csv')
submission.to_csv(sub_path, index=False)

print(f"Submission saved to: {sub_path}")
print(f"Submission shape: {submission.shape}")
print(f"\nSubmission distribution:")
print(submission['Irrigation_Need'].value_counts())
print(f"\nHead:")
submission.head(10)

In [ ]:
# Final Score Visualization
fig, ax = plt.subplots(figsize=(8, 5))

ax.text(0.5, 0.6, f'{final_score:.4f}',
        transform=ax.transAxes, fontsize=60, fontweight='bold',
        ha='center', va='center',
        color='#2ecc71' if final_score >= 0.99 else '#e74c3c')

ax.text(0.5, 0.3, 'Balanced Accuracy (OOF)',
        transform=ax.transAxes, fontsize=18, ha='center', va='center',
        color='#555555')

status = 'TARGET ACHIEVED!' if final_score >= 0.99 else 'Below target (0.99)'
ax.text(0.5, 0.15, status,
        transform=ax.transAxes, fontsize=14, ha='center', va='center',
        fontweight='bold',
        color='#2ecc71' if final_score >= 0.99 else '#e74c3c')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Final Model Performance', fontsize=20, fontweight='bold', pad=20)
fig.tight_layout()
plt.show()

print(f"\nFinal Balanced Accuracy: {final_score:.6f}")